# Kernel ICL transience, closed form: random features + exact ridge

What this measures: OOD test error of a Gaussian-kernel regressor on ICL linear regression, as a
function of **n_train** and of **task diversity T**. The kernel is approximated by random Fourier
features

$$\phi(z) = \sqrt{2/P}\,\cos(\Omega z + b),\qquad \Omega\sim N(0, I/w^2),\ b\sim U[0,2\pi],$$

and fit by an exact ridge solve on streamed $\Phi^\top\Phi$, $\Phi^\top y$. There is no training loop:
every point on a curve is a solve.

**The picture being tested.** With the task covariance whitened and the pool mean kept, the channels
a kernel learns, in eigenvalue order, do three different things out of distribution:

| channels | what they encode | OOD effect |
|---|---|---|
| `q` (degree 1) | the pool mean $\hat\mu$ | **up**: $\hat\mu$ is a pool artifact, the test mean is 0 |
| `y2q`, `Xyq`, `XXq` (degree 3) | the covariance, $=I$ here | **down**, below dMMSE |
| `y2XXq`, `yX_yX` (degree 5) | $\kappa_3$, the pool's skewness | **up** again |

So the OOD curve should rise slightly, dip below dMMSE, then climb back toward it. The horizontal
lines on every plot are the $n\to\infty$ floors of a kernel that has learned exactly the channels up to
that rung, computed for **the same pool** the model is trained on.

**The one hard limit.** A solve holds two P x P float64 blocks, so P <= 61,440 on an 80 GB A100. That
is enough at d=4, 4 pairs: the feature-count estimate for the $\kappa_3$ family (~66k) turned out to be
conservative, and runs at P = 61,440 showed the memorization rise -- at T=4 OOD went 0.94 -> 1.33 by
n = 262k while ID kept falling, and at T=8 one of two seeds rose at the very end of that grid. The
grid now runs to n ~ 1e6 (~17P) so the rise shows consistently.

**What the closed form cannot do** is reach the degree-3 floor during the dip: every mode it fits
costs finite-sample variance, including the ~17.5k degree-4 modes that carry no target content, so
the curve sits ~0.1-0.2 above the floors until n >> P -- by which point kappa_3 is coming in. The
dip is real but about half as deep as the floors say. One-pass SGD (the companion notebook) sees
fresh data every step, pays no such variance, and should reach the floor. Keep `PAIRS >= DIM`.

## Colab setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os

GIT_USER = "JoeyTurn"
GIT_REPO = "kernels_can_icl"

!git clone https://github.com/{GIT_USER}/{GIT_REPO}.git

sys.path.insert(0, "/content/kernels_can_icl")
sys.path.insert(0, "/content/kernels_can_icl/expts/rf_scaling")

RESULTPATH = "/content/drive/MyDrive/results"
os.environ["RESULTPATH"] = RESULTPATH
os.makedirs(RESULTPATH, exist_ok=True)

In [ ]:
import os
# Set before torch initializes CUDA: the solve needs two contiguous multi-GB P x P blocks.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import sys, math, time, pickle, dataclasses
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

if not os.path.exists("/content/kernels_can_icl"):
    ROOT = Path.cwd()
    while ROOT != ROOT.parent and not (ROOT / "kernels_can_icl" / "__init__.py").exists():
        ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT))
    sys.path.insert(0, str(ROOT / "expts" / "rf_scaling"))
    RESULTPATH = str(ROOT / "expts" / "rf_scaling" / "results")
    os.makedirs(RESULTPATH, exist_ok=True)

from kernels_can_icl.random_features import RFFKernelRegressor
import kicl_floor as KF
import transience_utils as TU

# A stale clone pairs this notebook with older modules; fail here, not an hour in.
_missing = [n for n in ("make_config", "build_ladder_engine", "ChannelSpectrum") if not hasattr(KF, n)]
_missing += [f"TU.{n}" for n in ("ChannelProbe", "plot_channel_probe", "probe_attribution")
             if not hasattr(TU, n)]
if _missing:
    raise RuntimeError(f"kicl_floor.py / transience_utils.py predate this notebook (missing {_missing}). "
                       "Run `!git -C /content/kernels_can_icl pull`, then Runtime > Restart.")
# The old solve() back-substituted with torch.cholesky_solve, which clones the factor: a third
# P x P block that OOMs an 80 GB card at P=61,440.
import inspect
if "torch.cholesky_solve(rhs" in inspect.getsource(RFFKernelRegressor.solve):
    raise RuntimeError("kernels_can_icl/random_features.py is stale (solve() uses cholesky_solve and "
                       "will OOM). Run `!git -C /content/kernels_can_icl pull`, then Runtime > Restart.")

# TF32 has ~10 mantissa bits: not enough for a near-ridgeless P x P solve.
torch.backends.cuda.matmul.allow_tf32 = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE.type == "cuda" else 0.0
print("device:", DEVICE, f"({torch.cuda.get_device_name(0)}, {GPU_GB:.1f} GB)" if GPU_GB
      else "(CPU -- runs the RF_SMOKE version)")

## Configuration

Defaults to the focused setting -- d=4, 4 pairs, eps^2=0.05, T = 4 and 8, three trials -- on the same
task pools as the SGD notebook (seeds are keyed on T), so the two can be compared run for run.
`DIM`, `PAIRS`, `EPS` and `T_LIST` change it; everything size-dependent follows, and all of it is part
of every result filename.

`RF_SMOKE=1` (automatic on CPU) runs a tiny version end to end in a few minutes.

In [ ]:
SMOKE = bool(int(os.environ.get("RF_SMOKE", "0"))) or DEVICE.type != "cuda"

DIM, PAIRS, EPS = 4, 4, 0.05     # the focused setting; keep PAIRS >= DIM
CFG = KF.make_config(DIM, PAIRS, EPS)   # sigma_x2 = 1/d; L_cfg = PAIRS + 1 (counts the query)
assert (CFG.dim, CFG.L, CFG.e) == (DIM, PAIRS, EPS), "stale kicl_floor.py -- git pull and restart"

POOL_KIND = "whiten_keepmean"    # Sigma_hat = I, mu_hat = the sample mean of the draws
WIDTH_FACTOR = 1.5               # x median||Z - Z'||: wide enough to separate degree 3 from 5 cleanly
SEED = 0

if SMOKE:
    T_LIST     = [1, 4, 16]
    CURVE_TS   = [4, 16]
    TRIALS     = 1
    P_FEATURES = 1_024
    N_GRID     = [2 ** k for k in range(4, 13)]           # 16 ... 4,096
    TEST_SZ, VAL_SZ, OOD_SZ = 2_048, 1_024, 4_096
    CHUNK, GEN_SHARD = 2_048, 4_096
else:
    T_LIST     = [4, 8]          # the focused setting; [1, 2, 4, 8, 16, 32, 64] for the full T sweep
    CURVE_TS   = T_LIST
    TRIALS     = 3               # same pools as the SGD notebook's three seeds (seeds are keyed on T)
    P_FEATURES = 61_440          # two P x P float64 blocks = 60 GB: the most an 80 GB card holds
    # Up to ~17P: the finite-n variance that holds the curve above the floors decays as n >> P, which
    # is where the kappa_3 family comes in.
    N_GRID     = [2 ** k for k in range(4, 21)]           # 16 ... 1,048,576, nested
    # OOD errors past degree 3 are heavy-tailed: at the kappa_3 level a 16k OOD set scatters +/-0.045 per
    # point (vs +/-0.01 at degree 3), a third of the rise. 131k brings that to ~0.015 for seconds of predict.
    TEST_SZ, VAL_SZ, OOD_SZ = 16_384, 8_192, 131_072
    CHUNK, GEN_SHARD = 4_096, 65_536

# The ridge. See "The ridge" below. Per-sample values (covariance units) are the ridge on the raw
# normal equations divided by n.
#   "absolute" : ONE ridge lambda on the raw normal equations (Phi^T Phi + lambda I) w = Phi^T y, i.e. kernel
#                ridge (K + lambda I) alpha = y with k(x,x) = 1, for every n and T (default). The standard
#                sample-independent ridge: the delta of the eigenlearning framework (Simon et al. 2021,
#                arXiv:2110.03922) and of the paper's eq. (15), and scikit-learn's Ridge / KernelRidge alpha.
#                At 4 pairs lambda = 1e-4 is 1.3-3.9 x kappa_0(n) over n = 16k-262k (where the old ID-validation
#                sweep landed) and ~1.1x at n = 2^20. RIDGE_ABS = None -> RIDGE_FIXED_FACTOR x kappa_0(2^20) x 2^20.
#   "scaled"   : RIDGE_FIXED_FACTOR x kappa_0(n) per sample at every n (no literature convention).
#   "fixed"    : one per-sample value at every n (the averaged-loss convention with fixed lambda). Anything
#                small enough to let kappa_3 in is ~ridgeless for n < ~30k: at 4 pairs, 2.6e-11 gave a 5x-null
#                peak at n ~ 16k whose tail hid the bottom of the U.
#   "sweep"    : per n, the best of kappa_0(n) x RIDGE_FACTORS on ID validation (5x the solves). It picks 3x
#                through the dip and drops to 1x late, and that drop is what made its rise.
RIDGE_MODE = "absolute"
RIDGE_ABS = 1e-4
RIDGE_FIXED = None
RIDGE_FIXED_FACTOR = 1.0
RIDGE_FACTORS = (3.0, 1.0, 0.3, 0.1, 0.01)

# Channel probe: pretraining contexts on which the trained model is projected onto the degree<=5 ICL
# features, to read off how much of each channel family it implements (see "What the model
# implements"). Degree-5 features are heavy-tailed: 100k contexts leave ~0.02 of noise in the
# coordinates, 400k ~0.003. Costs seconds per (T, trial) and ~0.2 GB. 0 turns it off.
N_PROBE = 20_000 if SMOKE else 400_000

# 16 bytes per P^2 element: the float64 accumulator plus cuSOLVER's working copy (no in-place GPU
# Cholesky). Drop P if the card is smaller than this was sized for.
_need = lambda P: (P * P * 16 + CHUNK * P * 8) / 1e9
if GPU_GB and _need(P_FEATURES) > 0.85 * GPU_GB:
    P_FEATURES = max(p for p in (16384, 24576, 32768, 40960, 49152, 57344, 61440)
                     if _need(p) < 0.85 * GPU_GB)
    print(f"card too small for the default P -- using P={P_FEATURES:,}")
print(CFG.describe())
print(f"SMOKE={SMOKE}  P={P_FEATURES:,}  n up to {N_GRID[-1]:,}  T={T_LIST}  trials={TRIALS}")
print(f"GPU memory ~{_need(P_FEATURES):.1f} GB" + (f" of {GPU_GB:.0f} GB" if GPU_GB else ""))

## Data and kernel width

`context_len = L_cfg` includes the query, so there are `PAIRS` in-context pairs and the flat context
`Z = [X.flatten, y.flatten, Xq]` has width `D = PAIRS(d+1) + d`. Noise is on every label including
the query, so MSE is against the noisy label -- as in the theory and the oracles. Every MSE is
divided by `norm2 = E[y_q^2]`.

The width is calibrated once, on a reference pool, and held fixed: letting it float with T would
change the model along the x-axis of the T-sweep.

In [ ]:
_td, _ = TU.make_pool_distr(CFG, 64, SEED, POOL_KIND, DEVICE)
_X = TU.gen_raw(_td, CFG, 4000, "pretrain")[0].double()
MEDIAN_DIST = float(torch.cdist(_X[:2000], _X[2000:]).flatten().median())
WIDTH = WIDTH_FACTOR * MEDIAN_DIST
D_FLAT = _X.shape[1]
del _X, _td
print(f"D = {D_FLAT}   median||Z-Z'|| = {MEDIAN_DIST:.3f}   width = {WIDTH:.3f} ({WIDTH_FACTOR}x)")
print(f"norm2 = {CFG.norm2:.4f}")

## Sizing: where each channel family should switch on

The Gaussian-kernel spectrum on this data is enumerable (two coordinate-variance classes), so the
eigenlearning conservation law gives the n at which each family reaches a learnability, and the
number of features whose span holds it. These are the vertical markers on the n-curves.

This is the i.i.d.-token spectrum and ignores the beta-induced correlation between tokens, so treat
the markers as where to look, not as a fit.

In [ ]:
SPEC = KF.ChannelSpectrum(CFG, WIDTH)
K3 = "y2XX"                                   # the kappa_3 family that carries the rise

print(f"lambda_1 (constant mode) = {SPEC.lam1:.3f}   ra = {SPEC.ra:.4f}   rb = {SPEC.rb:.4f}")
print(f"{'family':>18} {'lambda':>10} {'n @ L=.5':>11} {'n @ L=.9':>11} {'P for L=.5':>11} "
      f"{'L at n_max':>11}")
_n_eff = 1.0 / (1.0 / N_GRID[-1] + 1.0 / P_FEATURES)     # n and P both cap the resolvable modes
for kind, a, b in KF.LADDER:
    print(f"{KF.LADDER_LABELS[kind]:>18} {SPEC.family[kind]:>10.2e} {SPEC.n_for(kind, 0.5):>11,.0f} "
          f"{SPEC.n_for(kind, 0.9):>11,.0f} {SPEC.P_for(kind, 0.5):>11,.0f} "
          f"{SPEC.learnability(kind, _n_eff):>11.3f}")

_needP = SPEC.P_for(K3, 0.5)
print(f"\nkappa_3 family needs P ~ {_needP:,.0f} for learnability 0.5; configured P = {P_FEATURES:,}.")
if P_FEATURES < _needP:
    print("  -> P is below what the kappa_3 family needs. Expect the dip and a plateau, NOT the rise.\n"
          "     The SGD notebook has no P^2 memory and is the one that reaches the rise here.")

def kappa0(n):
    """Ridgeless kappa at n samples, with P capping the resolvable modes the way n does."""
    return SPEC.kappa(1.0 / (1.0 / n + 1.0 / P_FEATURES))

assert RIDGE_MODE in ("absolute", "scaled", "fixed", "sweep"), RIDGE_MODE
RIDGE_REL = RIDGE_FIXED if RIDGE_FIXED is not None else RIDGE_FIXED_FACTOR * kappa0(N_GRID[-1])
RIDGE_LAMBDA = RIDGE_ABS if RIDGE_ABS is not None else RIDGE_FIXED_FACTOR * kappa0(2 ** 20) * 2 ** 20

def ridge_grid(n):
    """Ridge(s) to solve at n, in covariance units (absolute ridge on Phi^T Phi = rel * n)."""
    if RIDGE_MODE == "absolute":
        return [RIDGE_LAMBDA / n]
    if RIDGE_MODE == "fixed":
        return [RIDGE_REL]
    if RIDGE_MODE == "scaled":
        return [RIDGE_FIXED_FACTOR * kappa0(n)]
    return [kappa0(n) * f for f in RIDGE_FACTORS]

print("kappa_0(n): " + "  ".join(f"n={n:,}: {kappa0(n):.1e}" for n in N_GRID[::max(1, len(N_GRID) // 5)]))
_n_end = 1.0 / (1.0 / N_GRID[-1] + 1.0 / P_FEATURES)
if RIDGE_MODE == "sweep":
    print(f"ridge: per-n sweep over kappa_0(n) x {RIDGE_FACTORS}, chosen on ID validation")
else:
    _r_end = ridge_grid(N_GRID[-1])[0]
    print(f"ridge ({RIDGE_MODE})" + (f": lambda = {RIDGE_LAMBDA:.3e} on Phi^T Phi" if RIDGE_MODE == "absolute" else "")
          + ", per sample (x kappa_0(n)):  " + "  ".join(
              f"n={n:,}: {ridge_grid(n)[0]:.1e} ({ridge_grid(n)[0] / kappa0(n):.2f}x)"
              for n in sorted({N_GRID[len(N_GRID) // 2], 2 ** 14, P_FEATURES, 2 ** 18, N_GRID[-1]})
              if N_GRID[0] <= n <= N_GRID[-1]))
    print(f"  kappa_3 learnability at n_max: {SPEC.learnability(K3, _n_end, _r_end):.2f} with this ridge, "
          f"{SPEC.learnability(K3, _n_end):.2f} ridgeless")

# vertical markers on the n-curves: where the dip should form and where the rise should start
MARKERS = {"Xyq L=.5": SPEC.n_for("Xy", 0.5), "XXq L=.9": SPEC.n_for("XX", 0.9),
           "kappa_3 L=.5": SPEC.n_for(K3, 0.5)}

In [ ]:
def estimate_runtime():
    td, _ = TU.make_pool_distr(CFG, 8, SEED, POOL_KIND, DEVICE)
    X, y = TU.gen_raw(td, CFG, min(CHUNK, 4096), "pretrain")[:2]
    r = RFFKernelRegressor(ridge=kappa0(4096) * 4096, kernel_width=WIDTH, num_features=P_FEATURES,
                           chunk=CHUNK, seed=0)
    r.accumulate(X[:256], y[:256])
    sync = torch.cuda.synchronize if DEVICE.type == "cuda" else (lambda: None)
    sync(); t0 = time.time(); r.accumulate(X, y); sync()
    per_row = (time.time() - t0) / X.shape[0]
    t0 = time.time(); r.solve(); sync()
    t_solve = time.time() - t0
    del r
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    n_solves = len(N_GRID) * len(ridge_grid(N_GRID[-1]))
    per_run = N_GRID[-1] * per_row + n_solves * t_solve * 1.15      # +15% for the predictions
    runs = len(T_LIST) * TRIALS
    print(f"accumulate {per_row * 1e6:.1f} us/row, solve {t_solve:.1f} s, {n_solves} solves per run")
    print(f"~{per_run / 60:.1f} min per (T, trial) x {runs} runs = ~{runs * per_run / 3600:.1f} h")

estimate_runtime()

## The ridge

Kernel ridge regression with one ridge for every n and every T (`RIDGE_MODE = "absolute"`):

$$(\Phi^\top\Phi + \lambda I)\,w = \Phi^\top y \quad\Longleftrightarrow\quad (K + \lambda I)\,\alpha = y, \qquad \lambda = 10^{-4},$$

with $K \approx \Phi\Phi^\top$ the Gaussian kernel, $k(x, x) = 1$. This is the standard
sample-independent ridge: the $\delta$ of the eigenlearning framework (Simon et al. 2021,
arXiv:2110.03922) and of the conservation law $\sum_i \lambda_i/(\lambda_i + \kappa) + \delta/\kappa = n$
the theory here uses. It is set once, before any data is seen, and nothing is selected along the
curve.

**Why this value.** Per sample the ridge is $\lambda/n$. Against $\kappa_0(n)$, the effective
regularization a ridgeless fit has at n, it is 1.3-3.9x over n = 16k-262k and ~1.1x at $n = 2^{20}$.
That is large enough to flatten the multiple-descent peaks and small enough at the end to let the
$\kappa_3$ family in: theory puts its learnability at $2^{20}$ at ~0.4. Over the dip it is where the
per-n ID-validation sweep landed on its own (3x). The sizing cell prints it at several n.

**What the other modes showed.** A near-ridgeless kernel regressor spikes each time n crosses the
number of modes in a band (the overfitting factor $E_0 = n/(n - \sum_i L_i^2)$ blows up). At 4 pairs
the band edges sit near n ~ 25, 300, 3k and 20k, and the random-feature interpolation threshold is
at n = P. A constant per-sample ridge (`"fixed"`, the averaged-loss convention) small enough to let
$\kappa_3$ in is ~ridgeless there: $2.6\times10^{-11}$ gave a 5x-null-error peak at n ~ 16k, and its tail
covered the bottom of the U. The per-n sweep (`"sweep"`) picked 3x through the dip and dropped to 1x
at the last points, and that drop, not n, made its rise.

## Main sweep

For each (T, trial):

1. draw the pool (seeded, on CPU) and use **that same pool** for the model, the ladder floors and
   both oracles;
2. draw ID-validation, ID-test and OOD-test sets from a dedicated seed;
3. stream training contexts, and at every n in `N_GRID` snapshot-solve at the ridge (non-destructive,
   so the accumulation continues; in "sweep" mode, at each candidate, keeping the best on ID
   validation);
4. with `N_PROBE > 0`, read every solve's channel content off its weights (see "What the model
   implements along the curve").

Checkpointed after every (T, trial); re-running the cell resumes.

In [ ]:
# Runs with the channel probe get their own file ("_probe"), so an older results file is never overwritten;
# so does each ridge setting.
RIDGE_TAG = ("sweep" if RIDGE_MODE == "sweep" else
             f"abs{RIDGE_LAMBDA:.2e}" if RIDGE_MODE == "absolute" else
             f"fixed{RIDGE_FIXED:.2e}" if RIDGE_MODE == "fixed" and RIDGE_FIXED is not None else
             f"{RIDGE_MODE}{RIDGE_FIXED_FACTOR:g}")
CKPT = os.path.join(RESULTPATH, f"rf_transience_closed_d{DIM}p{PAIRS}e{EPS:g}_P{P_FEATURES}_w{WIDTH_FACTOR}"
                                f"{'_probe' if N_PROBE else ''}_{RIDGE_TAG}{'_smoke' if SMOKE else ''}.pkl")
LADDER_ENGINE = KF.build_ladder_engine(CFG)

def _meta():
    return dict(config=dataclasses.asdict(CFG), dim=DIM, pairs=PAIRS, pool=POOL_KIND, width_factor=WIDTH_FACTOR,
                P=P_FEATURES, n_grid=N_GRID, ridge_mode=RIDGE_MODE, ridge_rel=RIDGE_REL, ridge_lambda=RIDGE_LAMBDA,
                ridge_fixed_factor=RIDGE_FIXED_FACTOR, ridge_factors=RIDGE_FACTORS,
                test_sz=TEST_SZ, val_sz=VAL_SZ, ood_sz=OOD_SZ, seed=SEED, smoke=SMOKE, n_probe=N_PROBE)

records = []
if os.path.exists(CKPT):
    with open(CKPT, "rb") as f:
        blob = pickle.load(f)
    if blob.get("meta") == _meta():
        records = blob["records"]
        print(f"resuming: {len(records)} (T, trial) runs already done")
    else:
        print("checkpoint is from a different configuration -- starting fresh")


def run_one(T, trial):
    seed = SEED + 1009 * trial + 17 * T          # keyed on T: the SGD notebook trains on the same pools
    td, betas = TU.make_pool_distr(CFG, T, seed, POOL_KIND, DEVICE)
    tests = TU.make_tests(td, CFG, {"val": (VAL_SZ, "pretrain"), "id": (TEST_SZ, "pretrain"),
                                    "ood": (OOD_SZ, "true")}, seed)
    rec = dict(T=T, trial=trial, seed=seed, betas=betas, grid={}, curve=[],
               **TU.references(tests, betas, CFG, LADDER_ENGINE))

    rff = RFFKernelRegressor(ridge=0.0, kernel_width=WIDTH, num_features=P_FEATURES,
                             chunk=CHUNK, seed=seed)
    probe = None
    produced = 0
    for n in N_GRID:
        while produced < n:
            m = min(GEN_SHARD, n - produced)
            Xs, ys = TU.gen_raw(td, CFG, m, "pretrain")[:2]
            rff.accumulate(Xs, ys)
            produced += m
            del Xs, ys
        if probe is None and N_PROBE:            # needs the feature draw, which the first accumulate makes
            probe = TU.ChannelProbe(td, CFG, betas, N_PROBE, seed, rff._features, tests, chunk=CHUNK)
            rec["probe"] = probe.summary()
        rec["grid"][n] = {}
        for rel in ridge_grid(n):
            rff.ridge = rel * n                  # covariance units -> absolute on Phi^T Phi
            rff.solve()                          # non-destructive: accumulation continues after
            preds = {s: rff.predict(tests[s]["X"], batch_size=4096) for s in ("val", "id", "ood")}
            res = {s: TU.nmse(preds[s], tests[s]["y"], CFG) for s in preds}
            if probe is not None:
                res["probe"] = probe.measure(rff.w, preds)
            rec["grid"][n][rel] = res
            del preds
        best = min(rec["grid"][n], key=lambda r: rec["grid"][n][r]["val"])
        rec["curve"].append(dict(n=n, ridge_rel=best, kappa0=kappa0(n), **rec["grid"][n][best]))
    del rff, tests, probe
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return rec


done = {(r["T"], r["trial"]) for r in records}
todo = [(T, tr) for tr in range(TRIALS) for T in T_LIST if (T, tr) not in done]
print(f"{len(todo)} runs to go -> {CKPT}")
t_all = time.time()
for i, (T, trial) in enumerate(todo):
    t0 = time.time()
    rec = run_one(T, trial)
    records.append(rec)
    with open(CKPT + ".tmp", "wb") as f:
        pickle.dump(dict(meta=_meta(), records=records), f)
    os.replace(CKPT + ".tmp", CKPT)
    ood = [c["ood"] for c in rec["curve"]]
    j = int(np.argmin(ood))
    left = (time.time() - t_all) / (i + 1) * (len(todo) - i - 1)
    print(f"T={T:>3} trial {trial}: OOD min {ood[j]:.3f} at n={rec['curve'][j]['n']:,}, "
          f"final {ood[-1]:.3f} | floor3 {rec['ladder']['XX'][1]:.3f}  "
          f"dMMSE {rec['oracles']['dmmse_ood']:.3f}  ({time.time() - t0:.0f}s, ~{left / 60:.0f} min left)")
    if rec.get("probe"):
        chk, lad = rec["probe"]["check"], rec["ladder"]
        k3 = [TU.implemented(rec["probe"], c["probe"]["coef"], TU.K3_KINDS) for c in (rec["curve"][j], rec["curve"][-1])]
        print(f"   probe: kappa_3 implemented {k3[0]:.2f} at the min -> {k3[1]:.2f} at the end; final OOD with "
              f"kappa_3 knocked out {rec['curve'][-1]['probe']['ko']['k3']['ood']:.3f}"
              f"  [check: target deg<=5 OOD {chk['deg5'][1]:.3f} vs floor {lad['yXdotX'][1]:.3f}]")
        if abs(chk["deg3"][0] - lad["XX"][0]) > 0.02 or abs(chk["deg5"][1] - lad["yXdotX"][1]) > 0.05:
            print("   WARNING: probe projections disagree with the theory floors -- do not trust the probe")
    if DEVICE.type == "cuda" and i == 0:
        print(f"   peak GPU {torch.cuda.max_memory_reserved() / 1e9:.1f} GB reserved")
print(f"done: {len(records)} records in {(time.time() - t_all) / 60:.1f} min")

## Error against n

One column per T. Dotted lines are the ladder floors (the kernel's $n\to\infty$ error if it had learned
exactly the channels up to that rung); dashed are dMMSE and Ridge; dash-dot are the predicted
switch-on points. The floors are for infinite n at infinite P, so the measured curve should approach
each one from above.

In [ ]:
FIGDIR = os.path.join(RESULTPATH, "figs")
os.makedirs(FIGDIR, exist_ok=True)
TAG = f"d{DIM}p{PAIRS}e{EPS:g}{'_smoke' if SMOKE else ''}"

TU.plot_n_curves(records, CFG, CURVE_TS, markers=MARKERS, xlabel="n_train",
                 title=f"closed form: {CFG.describe()}  |  P={P_FEATURES:,}, w={WIDTH_FACTOR}x median",
                 save=os.path.join(FIGDIR, f"rf_transience_closed_n_{TAG}.pdf"))
plt.show()
TU.summary_table(records, T_LIST)

## What the model implements along the curve

The floors and markers above are theory. This section measures directly, from each fitted model's
function, which channels it contains:

* **Implemented fraction.** The degree $\le 5$ ICL features are orthonormalized under the
  pretraining distribution, family by family in ladder order, so each family's coordinates hold only
  what the families before it do not. The Bayes (dMMSE) predictor on the pool has coordinates
  $a^\ast$ (an infinite-data kernel converges to it); the fitted model has $\hat a$. A family's
  implemented fraction is $\langle \hat a, a^\ast\rangle / \langle a^\ast, a^\ast\rangle$ over its
  coordinates: 0 before the model carries it, 1 once it does.
* **Knockout.** Subtract a family's coordinates from the model's own predictions and score it again.
  If the rise is the $\kappa_3$ family, the model with $\kappa_3$ knocked out should *not* rise OOD.
  Its ID error should also go *up*: that family is memorization, and memorization helps on the pool.

Both are read off the same solves as the curves above. The model is never refit. `probe_attribution`
gives the share of the OOD rise, from its minimum to the last n, that disappears when $\kappa_3$ is
removed.

In [ ]:
if N_PROBE:
    PROBE_N_FROM = SPEC.n_for("Xy", 0.9)          # look for the minimum after the ICL channel is in
    for T in CURVE_TS:
        TU.plot_channel_probe(records, CFG, T, xlabel="n_train",
                              title=f"closed form: {CFG.describe()}  |  P={P_FEATURES:,}, w={WIDTH_FACTOR}x median",
                              save=os.path.join(FIGDIR, f"rf_transience_closed_probe_T{T}_{TAG}.pdf"))
        plt.show()
    TU.probe_attribution(records, T_LIST, n_from=PROBE_N_FROM)
    print("self-check at the last n -- the measured fractions through the floor engine, vs what the "
          "model's projection scored (MISMATCH = the fractions are overlaps, not scalings):")
    TU.probe_consistency(records, CFG, LADDER_ENGINE, T_list=CURVE_TS)
else:
    print("N_PROBE = 0: no channel probe in these runs")

## Error against T

Figure-4 style, read off the same runs at two sample sizes: on the degree-3 plateau (after the dip
has formed, before the $\kappa_3$ family comes in) and at the largest n.

In [ ]:
_n_plateau = math.sqrt(SPEC.n_for("XX", 0.9) * SPEC.n_for(K3, 0.5))
_n_plateau = min(N_GRID, key=lambda n: abs(math.log(n) - math.log(_n_plateau)))
PICKS = [(f"RF, n={_n_plateau:,} (deg-3 plateau)", _n_plateau), (f"RF, n={N_GRID[-1]:,}", N_GRID[-1])]

TU.plot_T_sweep(records, CFG, T_LIST, PICKS,
                title=f"closed form: {CFG.describe()}  |  P={P_FEATURES:,}",
                save=os.path.join(FIGDIR, f"rf_transience_closed_T_{TAG}.pdf"))
plt.show()
print("figures ->", FIGDIR)